# Search Results Comparison Notebook

This notebook compares four search paths for the same set of names:

- `bing_search()` from the package
- direct DuckDuckGo via `_duckduckgo_search()`
- raw Bing HTML via `httpx` + `_extract_results()`
- raw Bing HTML via `pyppeteer` + `_extract_results()`

Use it to compare result counts, ranking, relevance scores, titles, snippets, and URLs side by side.

In [7]:
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import display

repo_root = Path.cwd()
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from institution_checker import INSTITUTION
from institution_checker.search import (
    bing_search,
    close_search_clients,
    _duckduckgo_search,
    _extract_results,
    _fetch_with_browser,
    _fetch_with_httpx,
    _prepare_ranked_results,
    _prioritize_target_hits,
)

print(f"Repo root: {repo_root}")
print(f"Institution: {INSTITUTION}")

Repo root: e:\Awards DB Code\step3_institution_checker
Institution: Purdue University


## Notes

- The `bing_search()` path is the package-level search entrypoint. Here it is called with `allow_fallback=False` so DDG is not mixed in.
- The raw `httpx` and `pyppeteer` paths both fetch Bing HTML directly and then parse it with the same `_extract_results()` helper for a fairer comparison.
- All methods use the same query template and `fetch_excerpts=False` to keep the comparison focused on search results rather than page scraping.

In [8]:
EXAMPLE_NAMES = [
    "Robert Duncan",
    "Mitch Daniels",
    "Akira Suzuki",
    "Geoffrey Hinton",
    "Albert Einstein",
]

BING_NUM_RESULTS = 10
DDG_DIRECT_NUM_RESULTS = 20
COMPARE_TOP_N = 5
QUERY_TEMPLATE = "{name} {institution}"

EXAMPLE_NAMES

['Robert Duncan',
 'Mitch Daniels',
 'Akira Suzuki',
 'Geoffrey Hinton',
 'Albert Einstein']

In [9]:
def build_query(name: str, institution: str = INSTITUTION) -> str:
    return QUERY_TEMPLATE.format(name=name, institution=institution).strip()


def prepare_results(results, name, limit):
    ranked = _prepare_ranked_results(results)
    ranked = _prioritize_target_hits(ranked, name, INSTITUTION)
    return ranked[:limit]


def summarize_result_row(name, method, elapsed_s, results, error=""):
    top = results[0] if results else {}
    top_signals = top.get("signals", {}) if top else {}
    return {
        "name": name,
        "method": method,
        "elapsed_s": round(elapsed_s, 2),
        "result_count": len(results),
        "top_score": top_signals.get("relevance_score", 0),
        "top_title": top.get("title", ""),
        "top_url": top.get("url", ""),
        "error": error,
    }


def flatten_results(name, method, query, results):
    rows = []
    for item in results[:COMPARE_TOP_N]:
        signals = item.get("signals", {}) or {}
        rows.append(
            {
                "name": name,
                "method": method,
                "query": query,
                "rank": item.get("rank"),
                "score": signals.get("relevance_score", 0),
                "has_person_name": signals.get("has_person_name"),
                "has_institution": signals.get("has_institution"),
                "has_explicit_connection": signals.get("has_explicit_connection"),
                "domain": signals.get("domain", ""),
                "title": item.get("title", ""),
                "snippet": item.get("snippet", ""),
                "url": item.get("url", ""),
            }
        )
    return rows


async def run_bing_search_method(name: str):
    query = build_query(name)
    results = await bing_search(
        query,
        num_results=BING_NUM_RESULTS,
        institution=INSTITUTION,
        person_name=name,
        debug=False,
        allow_fallback=False,
        fetch_excerpts=False,
    )
    return query, results


async def run_ddg_method(name: str):
    query = build_query(name)
    results = await _duckduckgo_search(
        query,
        institution=INSTITUTION,
        person_name=name,
        limit=DDG_DIRECT_NUM_RESULTS,
        debug=False,
    )
    return query, prepare_results(results, name, DDG_DIRECT_NUM_RESULTS)


async def run_httpx_method(name: str):
    query = build_query(name)
    html = await _fetch_with_httpx(query, count=max(BING_NUM_RESULTS * 5, 50), debug=False)
    results = _extract_results(
        html,
        institution=INSTITUTION,
        limit=BING_NUM_RESULTS * 2,
        person_name=name,
        debug=False,
    ) if html else []
    return query, prepare_results(results, name, BING_NUM_RESULTS)


async def run_pyppeteer_method(name: str):
    query = build_query(name)
    html = await _fetch_with_browser(query, count=max(BING_NUM_RESULTS * 5, 50), debug=False)
    results = _extract_results(
        html,
        institution=INSTITUTION,
        limit=BING_NUM_RESULTS * 2,
        person_name=name,
        debug=False,
    ) if html else []
    return query, prepare_results(results, name, BING_NUM_RESULTS)


METHODS = [
    ("bing_search", run_bing_search_method),
    ("ddg_direct", run_ddg_method),
    ("bing_httpx", run_httpx_method),
    ("bing_pyppeteer", run_pyppeteer_method),
]


async def compare_all(names):
    summary_rows = []
    detail_rows = []
    raw_results = {}

    for name in names:
        raw_results[name] = {}
        print(f"Running comparisons for: {name}")
        for method_name, runner in METHODS:
            started = time.perf_counter()
            try:
                query, results = await runner(name)
                elapsed = time.perf_counter() - started
                raw_results[name][method_name] = results
                summary_rows.append(summarize_result_row(name, method_name, elapsed, results))
                detail_rows.extend(flatten_results(name, method_name, query, results))
                print(f"  {method_name:<16} -> {len(results):>2} results in {elapsed:.2f}s")
            except Exception as exc:
                elapsed = time.perf_counter() - started
                raw_results[name][method_name] = []
                summary_rows.append(summarize_result_row(name, method_name, elapsed, [], error=str(exc)))
                print(f"  {method_name:<16} -> ERROR in {elapsed:.2f}s: {exc}")

    summary_df = pd.DataFrame(summary_rows)
    detail_df = pd.DataFrame(detail_rows)
    return summary_df, detail_df, raw_results

In [10]:
summary_df, comparison_df, raw_results = await compare_all(EXAMPLE_NAMES)

display(summary_df.sort_values(["name", "method"]).reset_index(drop=True))
display(
    comparison_df[
        [
            "name",
            "method",
            "rank",
            "score",
            "has_person_name",
            "has_institution",
            "has_explicit_connection",
            "title",
            "url",
        ]
    ].sort_values(["name", "method", "rank"]).reset_index(drop=True)
)

Running comparisons for: Robert Duncan
  bing_search      ->  0 results in 0.40s
  ddg_direct       -> 10 results in 3.68s
  bing_httpx       ->  0 results in 0.17s
  bing_pyppeteer   ->  7 results in 3.46s
Running comparisons for: Mitch Daniels
  bing_search      ->  0 results in 0.22s
  ddg_direct       -> 10 results in 3.54s
  bing_httpx       ->  0 results in 0.27s
  bing_pyppeteer   ->  0 results in 0.79s
Running comparisons for: Akira Suzuki
  bing_search      ->  0 results in 0.24s
  ddg_direct       -> 10 results in 3.62s
  bing_httpx       ->  0 results in 0.18s
  bing_pyppeteer   ->  7 results in 1.01s
Running comparisons for: Geoffrey Hinton
  bing_search      ->  0 results in 0.17s
  ddg_direct       -> 10 results in 3.87s
  bing_httpx       ->  7 results in 0.20s
  bing_pyppeteer   ->  7 results in 0.85s
Running comparisons for: Albert Einstein
  bing_search      ->  0 results in 0.34s
  ddg_direct       -> 10 results in 3.42s
  bing_httpx       ->  0 results in 0.10s
  bi

,name,method,elapsed_s,result_count,top_score,top_title,top_url,error
0,Akira Suzuki,bing_httpx,0.18,0,0,,,
1,Akira Suzuki,bing_pyppeteer,1.01,7,14,Akira Suzuki - Wikipedia,https://en.wikipedia.org/wiki/Akira_Suzuki,
2,Akira Suzuki,bing_search,0.24,0,0,,,
3,Akira Suzuki,ddg_direct,3.62,10,19,Akira Suzuki - A Superstar of Science,http://www.superstarsofscience.com/scientist/a...,
4,Albert Einstein,bing_httpx,0.10,0,0,,,
5,Albert Einstein,bing_pyppeteer,1.13,7,7,"Einstein , Albert , 1947 | Archives and Specia...",https://archives.lib.purdue.edu/repositories/2...,
6,Albert Einstein,bing_search,0.34,0,0,,,
7,Albert Einstein,ddg_direct,3.42,10,33,"Einstein, Albert, 1947 | Archives and Special ...",https://archives.lib.purdue.edu/repositories/2...,
8,Geoffrey Hinton,bing_httpx,0.20,7,12,Geoffrey Hinton - Wikipedia,https://en.wikipedia.org/wiki/Geoffrey_Hinton,
9,Geoffrey Hinton,bing_pyppeteer,0.85,7,12,Geoffrey Hinton - Wikipedia,https://en.wikipedia.org/wiki/Geoffrey_Hinton,


,name,method,rank,score,has_person_name,has_institution,has_explicit_connection,title,url
0,Akira Suzuki,bing_pyppeteer,1,14,True,True,False,Akira Suzuki - Wikipedia,https://en.wikipedia.org/wiki/Akira_Suzuki
1,Akira Suzuki,bing_pyppeteer,2,13,True,True,False,Akira Suzuki – Biographical - NobelPrize.org,https://www.nobelprize.org/prizes/chemistry/20...
2,Akira Suzuki,bing_pyppeteer,3,16,True,True,False,CV - Akira Suzuki | Lindau Mediatheque,https://mediatheque.lindau-nobel.org/laureates...
3,Akira Suzuki,bing_pyppeteer,4,13,True,True,False,"Akira Suzuki (born September 12, 1930), Japane...",https://prabook.com/web/akira.suzuki/201249
4,Akira Suzuki,bing_pyppeteer,6,14,True,True,False,Akira Suzuki - scientificlib.com,https://www.scientificlib.com/en/Chemistry/Bio...
5,Akira Suzuki,ddg_direct,1,19,True,True,False,Akira Suzuki - A Superstar of Science,http://www.superstarsofscience.com/scientist/a...
6,Akira Suzuki,ddg_direct,2,16,True,True,False,CV - Akira Suzuki | Lindau Mediatheque,https://mediatheque.lindau-nobel.org/laureates...
7,Akira Suzuki,ddg_direct,3,14,True,True,False,Akira Suzuki - scientificlib.com,https://www.scientificlib.com/en/Chemistry/Bio...
8,Akira Suzuki,ddg_direct,4,14,True,True,False,Akira Suzuki - Wikipedia,https://en.wikipedia.org/wiki/Akira_Suzuki
9,Akira Suzuki,ddg_direct,5,13,True,True,False,Akira Suzuki – Biographical - NobelPrize.org,https://www.nobelprize.org/prizes/chemistry/20...


In [11]:
NAME_TO_INSPECT = EXAMPLE_NAMES[0]

comparison_df[
    comparison_df["name"].eq(NAME_TO_INSPECT)
].sort_values(["method", "rank"]).reset_index(drop=True)

,name,method,query,rank,score,has_person_name,has_institution,has_explicit_connection,domain,title,snippet,url
0,Robert Duncan,bing_pyppeteer,Robert Duncan Purdue University,3,17,True,True,True,childandfamilyblog.com,Robert J. Duncan | Author | Child & Family Blog,Robert J. Duncan: Assistant Professor of Human...,https://childandfamilyblog.com/author/robert-j...
1,Robert Duncan,bing_pyppeteer,Robert Duncan Purdue University,4,11,True,True,False,www.purdue.edu,Influencing the Development of Social and ... ...,"Robert Duncan, Steps to Leaps Research Collabo...",https://www.purdue.edu/stepstoleaps/resources/...
2,Robert Duncan,bing_pyppeteer,Robert Duncan Purdue University,5,13,True,True,False,www.coursicle.com,Robert Duncan at Purdue University - Coursicle,Robert Duncan at Purdue University (Purdue) in...,https://www.coursicle.com/purdue/professors/Ro...
3,Robert Duncan,bing_pyppeteer,Robert Duncan Purdue University,6,16,True,True,False,boiler.courses,Robert J Duncan - boiler.courses,Instructor in the Human Development And Family...,https://boiler.courses/prof/815
4,Robert Duncan,bing_pyppeteer,Robert Duncan Purdue University,7,27,True,True,True,www.zoominfo.com,Robert Duncan - Assistant Professor at Purdue ...,Robert Duncan is an Assistant Professor at Pur...,https://www.zoominfo.com/p/Robert-Duncan/24719...
5,Robert Duncan,ddg_direct,Robert Duncan Purdue University,1,27,True,True,True,www.zoominfo.com,Robert Duncan - Assistant Professor at Purdue ...,Robert Duncan is an Assistant Professor at Pur...,https://www.zoominfo.com/p/Robert-Duncan/24719...
6,Robert Duncan,ddg_direct,Robert Duncan Purdue University,2,18,True,True,False,www.coursicle.com,Robert Duncan at Purdue University - Coursicle,Robert Duncan at Purdue University ( Purdue ) ...,https://www.coursicle.com/purdue/professors/Ro...
7,Robert Duncan,ddg_direct,Robert Duncan Purdue University,3,17,True,True,True,childandfamilyblog.com,Robert J. Duncan | Author | Child & Family Blog,Robert J. Duncan : Assistant Professor of Huma...,https://childandfamilyblog.com/author/robert-j...
8,Robert Duncan,ddg_direct,Robert Duncan Purdue University,4,16,True,True,False,boiler.courses,Robert J Duncan - boiler.courses,Instructor in the Human Development And Family...,https://boiler.courses/prof/815
9,Robert Duncan,ddg_direct,Robert Duncan Purdue University,5,16,True,True,True,www.linkedin.com,Trevor Duncan - Purdue University - Greater In...,Trevor Duncan . Student at Purdue University ....,https://www.linkedin.com/in/trevor-duncan-43b8...


In [12]:
await close_search_clients()